In [5]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score

import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier

# ============================
# LOAD DATA
# ============================
train = pd.read_csv("/content/train.csv")
test = pd.read_csv("/content/test.csv")

TARGET = "penghargaan_misi"

# ============================
# TIME PARSER
# ============================
def convert_time(s):
    if isinstance(s, float) or pd.isna(s):
        return np.nan
    m = re.match(r"(\d+)d\s+(\d+):(\d+):(\d+)\.(\d+)", s)
    if m:
        d, h, mi, sec, ms = map(int, m.groups())
        return d*86400 + h*3600 + mi*60 + sec + ms/1000
    return np.nan

time_cols_train = [c for c in train.columns if "waktu" in c.lower() or "durasi" in c.lower()]
time_cols_test = [c for c in test.columns if "waktu" in c.lower() or "durasi" in c.lower()]

for c in time_cols_train:
    train[c] = train[c].apply(convert_time)

for c in time_cols_test:
    test[c] = test[c].apply(convert_time)

# ============================
# LABEL ENCODE TARGET
# ============================
le = LabelEncoder()
train[TARGET] = le.fit_transform(train[TARGET])

# ============================
# CATEGORICAL DETECTION
# ============================
cat_cols = train.select_dtypes(include=["object"]).columns.tolist()
cat_cols = [c for c in cat_cols if c != TARGET]

# Encode only non-CatBoost models
enc_dict = {}
for c in cat_cols:
    enc = LabelEncoder()
    all_vals = pd.concat([train[c].astype(str), test[c].astype(str)], axis=0)
    enc.fit(all_vals)
    train[c] = enc.transform(train[c].astype(str))
    test[c] = enc.transform(test[c].astype(str))
    enc_dict[c] = enc

# ============================
# SPLIT DATA
# ============================
# Identify columns present in train but not in test (excluding the TARGET)
# These are likely target-leakage features or not available for prediction
leakage_cols = list(set(train.columns) - set(test.columns) - {TARGET})

X = train.drop(columns=[TARGET] + leakage_cols)
y = train[TARGET]
# Ensure X_test has the exact same columns as X
X_test = test[X.columns].copy()

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# ============================
# MODEL STORAGE
# ============================
test_pred_xgb = np.zeros((len(test), len(le.classes_)))
test_pred_lgb = np.zeros((len(test), len(le.classes_)))
test_pred_cat = np.zeros((len(test), len(le.classes_)))

oof_pred = np.zeros(len(train))

# ===================================================
# PARAMETER MODEL OPTIMAL LEVEL 3
# ===================================================

xgb_params = {
    "objective": "multi:softprob",
    "num_class": len(le.classes_),
    "max_depth": 8,
    "learning_rate": 0.05,
    "subsample": 0.9,
    "colsample_bytree": 0.9,
    "n_estimators": 1200,
    "tree_method": "hist"
}

lgb_params = {
    "objective": "multiclass",
    "num_class": len(le.classes_),
    "max_depth": -1,
    "learning_rate": 0.03,
    "n_estimators": 1500,
    "subsample": 0.9,
    "colsample_bytree": 0.8
}

cat_params = {
    "iterations": 1200,
    "learning_rate": 0.03,
    "depth": 8,
    "loss_function": "MultiClass",
    "verbose": False
}

# ============================
# TRAINING LOOP
# ============================
fold = 1
for train_idx, val_idx in kf.split(X, y):

    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    # -----------------------------
    # XGBOOST
    # -----------------------------
    model_xgb = xgb.XGBClassifier(**xgb_params)
    model_xgb.fit(X_tr, y_tr)
    val_px = model_xgb.predict_proba(X_val)
    test_pred_xgb += model_xgb.predict_proba(X_test)

    # -----------------------------
    # LIGHTGBM
    # -----------------------------
    model_lgb = lgb.LGBMClassifier(**lgb_params)
    model_lgb.fit(X_tr, y_tr)
    val_pl = model_lgb.predict_proba(X_val)
    test_pred_lgb += model_lgb.predict_proba(X_test)

    # -----------------------------
    # CATBOOST (native categorical)
    # -----------------------------
    model_cat = CatBoostClassifier(**cat_params)
    model_cat.fit(
        X_tr, y_tr,
        eval_set=(X_val, y_val)
    )
    val_pc = model_cat.predict_proba(X_val)
    test_pred_cat += model_cat.predict_proba(X_test)

    # OOF = majority vote
    val_stack = (val_px + val_pl + val_pc) / 3
    oof_pred[val_idx] = np.argmax(val_stack, axis=1)

    f1 = f1_score(y_val, oof_pred[val_idx], average="macro")
    print(f"FOLD {fold} F1-macro = {f1:.4f}")
    fold += 1

# ============================
# FINAL PREDICTION
# ============================
test_final = (test_pred_xgb + test_pred_lgb + test_pred_cat) / (3 * kf.n_splits)
test_label = np.argmax(test_final, axis=1)
test_label = le.inverse_transform(test_label)

submission = pd.DataFrame({
    "id": test["id"],
    "penghargaan_misi": test_label
})

submission.to_csv("submission_level3_ensemble.csv", index=False)
print("\nSaved as submission_level3_ensemble.csv")

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001411 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2328
[LightGBM] [Info] Number of data points in the train set: 4406, number of used features: 38
[LightGBM] [Info] Start training from score -1.111404
[LightGBM] [Info] Start training from score -3.053184
[LightGBM] [Info] Start training from score -1.015467
[LightGBM] [Info] Start training from score -1.786779
[LightGBM] [Info] Start training from score -2.364857
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[Li